Imports

In [1]:
import tensorflow as tf
import keras as kr
from keras.src.callbacks import ModelCheckpoint

2025-03-25 00:13:13.720327: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742857993.860397    7779 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742857993.902908    7779 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1742857994.219534    7779 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1742857994.219656    7779 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1742857994.219659    7779 computation_placer.cc:177] computation placer alr

GPU

In [ ]:
gpu_list = tf.config.list_physical_devices('GPU')
print("Num GPUs Available: ", len(gpu_list))
tf.config.experimental.set_memory_growth(gpu_list[0], True)

Init

In [ ]:
IMAGE_WIDTH = 32
IMAGE_HEIGHT = 32
BATCH_SIZE = 16

#            0        1       2       3        4        5
CLASSES = ["None", "Left", "Right", "Top", "Bottom", "Front"]
NUM_OF_CLASSES = len(CLASSES)

Model

In [ ]:
#MAX acc = 0.76663
model = kr.Sequential()
model.add(kr.layers.Input(shape=(IMAGE_WIDTH, IMAGE_HEIGHT, 3)))

model.add(kr.layers.Rescaling(1./255)) #Normalizacia

model.add(kr.layers.Conv2D(filters=512, kernel_size=3, activation='relu', padding='same'))
#model.add((kr.layers.BatchNormalization()))

model.add(kr.layers.Conv2D(filters=256, kernel_size=3, activation='relu', padding='same'))
#model.add((kr.layers.BatchNormalization()))
model.add((kr.layers.MaxPool2D((2, 2))))
model.add(kr.layers.Dropout(0.1))

model.add(kr.layers.Conv2D(filters=128, kernel_size=3, activation='relu', padding='same'))
#model.add((kr.layers.BatchNormalization()))
model.add((kr.layers.MaxPool2D((2, 2))))
model.add(kr.layers.Dropout(0.1))

model.add(kr.layers.Conv2D(filters=64, kernel_size=3, activation='relu', padding='same'))
model.add((kr.layers.MaxPool2D((2, 2))))
model.add(kr.layers.Dropout(0.1))

model.add(kr.layers.Conv2D(filters=32, kernel_size=3, activation='relu', padding='same'))
model.add((kr.layers.MaxPool2D((2, 2))))
model.add(kr.layers.Dropout(0.1))

model.add(kr.layers.Flatten())

model.add(kr.layers.Dense(units=32, activation='relu'))
model.add(kr.layers.Dense(units=NUM_OF_CLASSES, activation='softmax'))

model.compile(
    loss=kr.losses.SparseCategoricalCrossentropy(),
    optimizer=kr.optimizers.Adam(),
    metrics=['accuracy']
)

model.summary()

Dataset

In [ ]:
dataset_path = "./organized_images"

train_dataset:tf.data.Dataset = kr.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    batch_size=BATCH_SIZE,
    shuffle=True
)

validation_dataset:tf.data.Dataset = kr.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    batch_size=BATCH_SIZE
)

AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

Callbacks

In [ ]:
weights_filepath = './model.weights.h5'
weights_save = ModelCheckpoint(
    filepath=weights_filepath,
    save_best_only=True,
    save_weights_only=True,
    #???? s viac datami asi mozem odkomentovat inak nechat tak
    monitor='val_accuracy',
    save_freq='epoch',
    mode='max',
    verbose=1
)

earty_stopping = kr.callbacks.EarlyStopping(monitor="val_loss", patience=10)

learning_rate = kr.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.2,
    patience=3,
    min_lr=0.00001
)

Load weights

In [ ]:
try:
    model.load_weights(weights_filepath)
except:
    print("cant load weights")

Fit model

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=50,
    #shuffle=True,
    verbose=1,
    callbacks=[weights_save]
)